# SAFE binary v7 — 금전 사기 커버리지 보강 (합성+사람승인 → 재학습 → fresh blind v10)

v6 파이프라인 ⊇ + 앞단에 **사기 데이터 생성**(Gemini 합성 → LLM 검수 → 사람 승인·승격).
v4 파일럿은 CPU 런타임에서도 가능하며, 재학습부터 GPU가 필요하다. GEMINI_API_KEY, 고정 SHA eval 3파일, PAN12 정본이 필요하다.
v9는 합성 누수 검사에 이미 사용된 회귀셋이다. 최종 1회 평가는 별도 작성·봉인한 v10만 사용한다.

In [ ]:
# 1. 현재 런타임·저장소·브랜치 확인 (파일럿은 CPU 가능)
import subprocess, sys, json, hashlib, re, shutil, collections, os, tempfile
from pathlib import Path
import torch
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
else: print('CPU 런타임: v4 파일럿은 가능하며, 재학습 전에 GPU 런타임으로 연결해야 합니다.')
BRANCH='feature/safe-scam-augmentation'
REMOTE='https://github.com/threeGuineas/thisabled-ai.git'
cwd=Path.cwd().resolve()
candidates=[cwd,*cwd.parents,cwd/'thisabled-ai']
REPO=next((p for p in candidates if (p/'.git').exists()),None)
if REPO is None:
    clone_parent=Path('/content') if Path('/content').exists() else cwd
    REPO=clone_parent/'thisabled-ai'
    assert not REPO.exists(), f'clone 대상이 이미 있으나 git 저장소가 아닙니다: {REPO}'
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)],check=True)
assert (REPO/'.git').exists(), f'저장소 준비 실패: {REPO}'
current_branch=subprocess.check_output(['git','branch','--show-current'],cwd=REPO,text=True).strip()
assert current_branch==BRANCH, f'현재 브랜치 {current_branch!r} != {BRANCH!r}'
subprocess.run(['git','pull','--ff-only','origin',BRANCH],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],cwd=REPO,check=True)
sys.path.insert(0,str(REPO))
def write_json_atomic(path,payload):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.NamedTemporaryFile(mode='w',encoding='utf-8',dir=path.parent,delete=False) as temp:
        json.dump(payload,temp,ensure_ascii=False,indent=2); temp.write('\n'); temp.flush(); os.fsync(temp.fileno()); temporary=Path(temp.name)
    os.replace(temporary,path)
def next_versioned_path(path):
    path=Path(path)
    if not path.exists(): return path
    for version in range(2,1000):
        candidate=path.with_name(f'{path.stem}_v{version}{path.suffix}')
        if not candidate.exists(): return candidate
    raise RuntimeError(f'사용 가능한 버전 파일명 없음: {path}')

In [ ]:
# 2. data_bundle_eval.zip + 별도 PAN12 정본 — VS Code 탐색기 경로 우선, 필요한 파일만 검증·반영
import io, zipfile, stat
from pathlib import PurePosixPath
REQUIRED_BUNDLE_FILES={'data/eval/aihub_train.jsonl':'3ebebb6e16468d47d603c22d659316514d45c8023b37ebb036b1411093f60e36',
    'data/eval/aihub_real_holdout.jsonl':'c7546a286942bde7c0bf7c69fc13b7257e6ec85daafeec4ee3d955f316275c51',
    'data/eval/beep_real_holdout.jsonl':'7062448ade6e244904a88674829867f63d95eac190bc4b7070f255bcda30c6fc'}
ALLOWED_BUNDLE_ENTRIES=set(REQUIRED_BUNDLE_FILES)|{'data/','data/eval/'}
def extract_data_bundle(bundle):
    with zipfile.ZipFile(bundle) as z:
        infos=z.infolist(); names=set(); total_size=0; repo_root=REPO.resolve()
        eval_root=(REPO/'data/eval').resolve(); assert eval_root==repo_root/'data'/'eval', 'data/eval symlink 경로 거부'
        for info in infos:
            name=info.filename; pure=PurePosixPath(name)
            assert name and '\\' not in name and not pure.is_absolute() and '..' not in pure.parts, f'위험 ZIP 경로: {name}'
            assert name in ALLOWED_BUNDLE_ENTRIES, f'허용되지 않은 ZIP 항목: {name}'
            assert name not in names, f'중복 ZIP 경로: {name}'; names.add(name)
            mode=(info.external_attr>>16)&0o170000
            assert mode!=stat.S_IFLNK and not (info.flag_bits&1), f'symlink/암호화 항목 거부: {name}'
            total_size+=info.file_size; assert total_size<=5*1024**3, '압축 해제 크기 5GiB 초과'
        missing=sorted(set(REQUIRED_BUNDLE_FILES)-names); assert not missing, f'필수 파일 누락: {missing}'
        for name,expected_sha in sorted(REQUIRED_BUNDLE_FILES.items()):
            info=z.getinfo(name); assert not info.is_dir(), f'필수 파일이 디렉터리임: {name}'
            target=(REPO/Path(*PurePosixPath(name).parts)).resolve(); assert target.parent==eval_root, f'추출 경로 이탈: {target}'
            target.parent.mkdir(parents=True,exist_ok=True); temp_path=None
            try:
                with z.open(info) as source, tempfile.NamedTemporaryFile(dir=target.parent,delete=False) as temp:
                    temp_path=Path(temp.name); shutil.copyfileobj(source,temp); temp.flush(); os.fsync(temp.fileno())
                assert hashlib.sha256(temp_path.read_bytes()).hexdigest()==expected_sha, f'eval 정본 SHA 불일치: {name}'
                os.replace(temp_path,target); temp_path=None
            finally:
                if temp_path is not None: temp_path.unlink(missing_ok=True)
            assert hashlib.sha256(target.read_bytes()).hexdigest()==expected_sha, f'eval 저장 SHA 불일치: {name}'
    print('안전 추출 완료:',sorted(REQUIRED_BUNDLE_FILES))
required_targets={name:REPO/name for name in REQUIRED_BUNDLE_FILES}
if all(path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==REQUIRED_BUNDLE_FILES[name] for name,path in required_targets.items()):
    print('기존 eval 정본 3개 SHA 확인 완료')
else:
    bundle_candidates=[cwd/'data_bundle_eval.zip',REPO/'data_bundle_eval.zip',REPO.parent/'data_bundle_eval.zip']
    bundle_path=next((p for p in bundle_candidates if p.is_file()),None)
    if bundle_path is not None:
        print('VS Code/현재 작업 경로 eval 전용 번들 사용:',bundle_path); extract_data_bundle(bundle_path)
    else:
        print('VS Code 탐색기로 data_bundle_eval.zip을 현재 작업 폴더나 저장소 루트에 둔 뒤 이 셀을 다시 실행하세요.')
        try:
            import ipywidgets as widgets
            from IPython.display import display
            uploader=widgets.FileUpload(accept='.zip',multiple=False,description='data_bundle_eval.zip 선택')
            def on_upload(change):
                value=uploader.value
                if not value: return
                item=next(iter(value.values())) if isinstance(value,dict) else value[0]
                extract_data_bundle(io.BytesIO(bytes(item['content'])))
            uploader.observe(on_upload,names='value'); display(uploader)
        except ImportError:
            print('widget도 없습니다. VS Code 탐색기로 data_bundle_eval.zip을 배치하세요.')
PAN12_SHA='93c4533477a80c8b3096fc4b86ad445f6fa302ab9fc5225f8790d4c5c260bfae'
PAN12_SOURCE_PATH_RAW=''  # 선택: 본생성 후 데이터 빌드 전까지 VS Code로 옮긴 pan12_translated.jsonl 경로
pan12_target=REPO/'data/synthetic/pan12_translated.jsonl'
PAN12_READY=False
if pan12_target.exists():
    assert hashlib.sha256(pan12_target.read_bytes()).hexdigest()==PAN12_SHA, 'PAN12 정본 SHA 불일치'
    PAN12_READY=True
elif PAN12_SOURCE_PATH_RAW.strip():
    pan12_source=Path(PAN12_SOURCE_PATH_RAW).expanduser().resolve(); assert pan12_source.is_file() and hashlib.sha256(pan12_source.read_bytes()).hexdigest()==PAN12_SHA
    pan12_target.parent.mkdir(parents=True,exist_ok=True)
    with pan12_source.open('rb') as source, tempfile.NamedTemporaryFile(dir=pan12_target.parent,delete=False) as temp:
        shutil.copyfileobj(source,temp); temp.flush(); os.fsync(temp.fileno()); pan12_temp=Path(temp.name)
    os.replace(pan12_temp,pan12_target)
    PAN12_READY=True
if PAN12_READY: print('PAN12 정본 SHA 확인:',PAN12_SHA)
else: print('PAN12 미준비: v4 파일럿은 실행할 수 있습니다. Cell 4 데이터 빌드 전에는 PAN12_SOURCE_PATH_RAW를 입력하고 Cell 2를 다시 실행하세요.')

In [ ]:
# 2b. v4 파일럿 — 유형별 5건 생성·자동 검수 (CPU Colab 가능, 본생성·승격 아님)
from getpass import getpass
PILOT_RUN_DATE='20260728'  # 최초 실행일로 고정. 429 재개 때 바꾸지 않음
PILOT_VERSION=4
assert re.fullmatch(r'[0-9]{8}',PILOT_RUN_DATE) and PILOT_VERSION==4
pilot_output_dir=REPO/'outputs/10_thisabled-ai/데이터'; pilot_output_dir.mkdir(parents=True,exist_ok=True)
pilot_candidate=pilot_output_dir/f'{PILOT_RUN_DATE}_thisabled_사기파일럿후보_v{PILOT_VERSION}.jsonl'
pilot_cache=pilot_output_dir/f'{PILOT_RUN_DATE}_thisabled_사기파일럿체크포인트_v{PILOT_VERSION}.json'
pilot_report=pilot_output_dir/f'{PILOT_RUN_DATE}_thisabled_사기파일럿검수보고서_v{PILOT_VERSION}.json'
pilot_forbidden=['data/eval/aihub_real_holdout.jsonl','data/eval/beep_real_holdout.jsonl',
    'data/synthetic/emergency/3a/val.jsonl','data/synthetic/emergency/3a/test.jsonl',
    *[f'tests/fixtures/safe_blind_v{i}.jsonl' for i in range(1,10)]]
missing_pilot_inputs=[path for path in pilot_forbidden if not (REPO/path).is_file() or (REPO/path).stat().st_size==0]
assert not missing_pilot_inputs, f'파일럿 forbidden 입력 누락·빈 파일: {missing_pilot_inputs}'
if pilot_report.exists():
    previous_pilot_report=json.loads(pilot_report.read_text())
    previous_pilot_status=str(previous_pilot_report.get('status',''))
    assert previous_pilot_status!='complete', 'v4 파일럿이 이미 완료됨 — 재실행하지 말고 사람 검수로 이동하세요.'
    assert not previous_pilot_status.startswith('failed_'), 'v4 자동 게이트 실패 — 캐시를 재사용하지 말고 새 버전 파일명으로 다시 생성하세요.'
assert not pilot_candidate.exists(), '완료 보고서 없는 기존 v4 후보 충돌 — 원인을 확인하기 전 덮어쓰지 않습니다.'
pilot_api_key=getpass('GEMINI_API_KEY: ').strip()  # 출력·전역 os.environ에 저장하지 않음
assert pilot_api_key, 'GEMINI_API_KEY가 필요합니다.'
pilot_env=os.environ.copy(); pilot_env['GEMINI_API_KEY']=pilot_api_key
pilot_cmd=[sys.executable,'scripts/build_scam_dataset.py','--per-subtype','5',
    '--synth-model','gemini-3.5-flash','--verify-model','gemini-3.5-flash',
    '--request-interval','15','--min-retention-ratio','0.8',
    '--out',str(pilot_candidate),'--synthesis-cache',str(pilot_cache),
    '--verification-report',str(pilot_report)]
for path in pilot_forbidden: pilot_cmd+=['--forbidden',path]
pilot_output_tail=collections.deque(maxlen=80)
try:
    pilot_process=subprocess.Popen(pilot_cmd,cwd=REPO,env=pilot_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    assert pilot_process.stdout is not None
    for pilot_line in pilot_process.stdout:
        print(pilot_line,end=''); pilot_output_tail.append(pilot_line)
    pilot_returncode=pilot_process.wait()
finally:
    pilot_env.pop('GEMINI_API_KEY',None); pilot_api_key=None
if pilot_returncode:
    pilot_diagnostic=''.join(pilot_output_tail).strip() or '(자식 프로세스 출력 없음)'
    raise RuntimeError(f'v4 파일럿 실패 (exit={pilot_returncode}). 체크포인트를 삭제하지 말고 아래 진단을 전달하세요.\n{pilot_diagnostic}')
pilot_payload=json.loads(pilot_report.read_text()); assert pilot_payload.get('status')=='complete', pilot_payload.get('status')
pilot_sha=hashlib.sha256(pilot_candidate.read_bytes()).hexdigest()
assert pilot_payload.get('final_output',{}).get('sha256')==pilot_sha, 'v4 후보와 검수 보고서 SHA 불일치'
pilot_rows=[json.loads(line) for line in pilot_candidate.read_text().splitlines() if line]
pilot_subtype_counts=collections.Counter(row['subtype'] for row in pilot_rows)
assert len(pilot_subtype_counts)==9 and all(count>=4 for count in pilot_subtype_counts.values()), 'v4 subtype별 80% 보존 게이트 미달'
print('v4 candidate',len(pilot_rows),'| label',dict(collections.Counter(row['label'] for row in pilot_rows)))
print('subtype',dict(sorted(pilot_subtype_counts.items())))
for subtype in sorted(pilot_subtype_counts):
    print(f'\n[{subtype}]')
    for row in [item for item in pilot_rows if item['subtype']==subtype]: print('-',row['text'])
print('\n후보:',pilot_candidate,'\n체크포인트:',pilot_cache,'\n검수 보고서:',pilot_report,'\n후보 SHA-256:',pilot_sha)
print('다음 단계: 전 행을 keep/revise/drop으로 사람 검수. drop=0이고 revise 원인을 해소하기 전에는 Cell 3 본생성을 실행하지 않습니다.')

In [ ]:
# 3. 금전 사기 본생성 후보 (승인 전에는 canonical train 경로를 건드리지 않음)
from datetime import datetime
from getpass import getpass
from zoneinfo import ZoneInfo
RUN_DATE=''     # 최초 실행일(YYYYMMDD)을 한 번 입력하고 자정이 지나도 429 재개 시 바꾸지 않음
RUN_VERSION=1   # 같은 날 새 품질 라운드만 2, 3으로 올림. 같은 값은 429 재개에만 사용
assert re.fullmatch(r'[0-9]{8}',RUN_DATE) and type(RUN_VERSION) is int and RUN_VERSION>=1
run_date=RUN_DATE; version_suffix='' if RUN_VERSION==1 else f'_v{RUN_VERSION}'
data_output_dir=REPO/'outputs/10_thisabled-ai/데이터'
candidate_path=data_output_dir/f'{run_date}_thisabled_사기본생성후보{version_suffix}.jsonl'
synthesis_cache=data_output_dir/f'{run_date}_thisabled_사기본생성체크포인트{version_suffix}.json'
verification_report=data_output_dir/f'{run_date}_thisabled_사기본생성검수보고서{version_suffix}.json'
gemini_api_key=getpass('GEMINI_API_KEY: ')  # subprocess 전용 환경으로만 전달
child_env=os.environ.copy()
child_env['GEMINI_API_KEY']=gemini_api_key
cmd=[sys.executable,'scripts/build_scam_dataset.py','--per-subtype','40',
     '--synth-model','gemini-3.5-flash','--verify-model','gemini-3.5-flash',
     '--request-interval','15','--min-retention-ratio','0.8',
     '--out',str(candidate_path),'--synthesis-cache',str(synthesis_cache),
     '--verification-report',str(verification_report)]
scam_forbidden=['data/eval/aihub_real_holdout.jsonl','data/eval/beep_real_holdout.jsonl',
    'data/synthetic/emergency/3a/val.jsonl','data/synthetic/emergency/3a/test.jsonl',
    *[f'tests/fixtures/safe_blind_v{i}.jsonl' for i in range(1,10)]]
for path in scam_forbidden: cmd+=['--forbidden',path]
try:
    subprocess.run(cmd,cwd=REPO,check=True,env=child_env)
finally:
    child_env.pop('GEMINI_API_KEY',None)
    gemini_api_key=None
scam=[json.loads(x) for x in candidate_path.read_text().splitlines() if x]
subtype_counts=collections.Counter(x['subtype'] for x in scam)
print('scam candidate',len(scam),'| label',dict(collections.Counter(x['label'] for x in scam)))
print('subtype',dict(subtype_counts))
print('합성 체크포인트:',synthesis_cache)
print('자동 검수 보고서:',verification_report)
assert all(n>=32 for n in subtype_counts.values()) and len(subtype_counts)==9, 'subtype별 80% 보존 게이트 미달'
for subtype in sorted(subtype_counts):
    print(f'\n[{subtype}]')
    for row in [x for x in scam if x['subtype']==subtype][:5]: print('-',row['text'])
verification_payload=json.loads(verification_report.read_text())
SCAM_ARTIFACT_SHA=hashlib.sha256(candidate_path.read_bytes()).hexdigest()
REVIEW_ARTIFACT_SHA=hashlib.sha256(verification_report.read_bytes()).hexdigest()
assert verification_payload['status']=='complete', verification_payload.get('status')
assert verification_payload['final_output']['sha256']==SCAM_ARTIFACT_SHA, '후보와 검수 보고서 SHA 불일치'
print('승인 대상 scam SHA-256:',SCAM_ARTIFACT_SHA)
print('승인 대상 report SHA-256:',REVIEW_ARTIFACT_SHA)

In [ ]:
# 3b. 사람 품질 승인·원자 승격 — 생성 셀을 재실행하지 말고 이 셀만 수정·실행
HUMAN_QUALITY_APPROVED=False  # 검수 보고서와 subtype별 표본을 확인한 뒤에만 True
APPROVED_SCAM_SHA=''          # 위 셀에서 출력된 scam SHA-256을 붙여넣기
APPROVED_REPORT_SHA=''        # 위 셀에서 출력된 report SHA-256을 붙여넣기
assert HUMAN_QUALITY_APPROVED, '사람 품질 승인 전에는 데이터 빌드·재학습을 진행하지 않습니다.'
assert APPROVED_SCAM_SHA==SCAM_ARTIFACT_SHA and APPROVED_REPORT_SHA==REVIEW_ARTIFACT_SHA, '승인 SHA 입력 불일치'
assert hashlib.sha256(candidate_path.read_bytes()).hexdigest()==APPROVED_SCAM_SHA, '승인 전 후보 변경 감지'
assert hashlib.sha256(verification_report.read_bytes()).hexdigest()==APPROVED_REPORT_SHA, '승인 후 검수 보고서 변경 감지'
approved_report=json.loads(verification_report.read_text())
assert approved_report['status']=='complete' and approved_report['final_output']['sha256']==APPROVED_SCAM_SHA, '완료 검수 보고서가 아님'
scam_path=REPO/'data/synthetic/scam/train.jsonl'; approval_manifest=REPO/'data/synthetic/scam/approval.json'
scam_path.parent.mkdir(parents=True,exist_ok=True)
with candidate_path.open('rb') as source, tempfile.NamedTemporaryFile(dir=scam_path.parent,delete=False) as temp:
    shutil.copyfileobj(source,temp); temp.flush(); os.fsync(temp.fileno()); promoted_temp=Path(temp.name)
os.replace(promoted_temp,scam_path)
approval_payload={'schema_version':1,'human_approved':True,'approved_at':datetime.now(ZoneInfo('Asia/Seoul')).isoformat(),
    'candidate_path':str(candidate_path.relative_to(REPO)),'dataset_path':str(scam_path.relative_to(REPO)),
    'dataset_sha256':APPROVED_SCAM_SHA,'row_count':len(scam),
    'verification_report_path':str(verification_report.relative_to(REPO)),
    'verification_report_sha256':APPROVED_REPORT_SHA}
with tempfile.NamedTemporaryFile(mode='w',encoding='utf-8',dir=approval_manifest.parent,delete=False) as temp:
    json.dump(approval_payload,temp,ensure_ascii=False,indent=2); temp.write('\n'); temp.flush(); os.fsync(temp.fileno()); manifest_temp=Path(temp.name)
os.replace(manifest_temp,approval_manifest)
def assert_approved_scam_artifacts():
    assert HUMAN_QUALITY_APPROVED, '사람 품질 승인 필요'
    assert hashlib.sha256(candidate_path.read_bytes()).hexdigest()==APPROVED_SCAM_SHA, '승인 후보 변경 감지'
    assert hashlib.sha256(scam_path.read_bytes()).hexdigest()==APPROVED_SCAM_SHA, '승격 train 변경 감지'
    assert hashlib.sha256(verification_report.read_bytes()).hexdigest()==APPROVED_REPORT_SHA, '검수 보고서 변경 감지'
    manifest=json.loads(approval_manifest.read_text()); assert manifest==approval_payload, '승인 manifest 변경 감지'
    report=json.loads(verification_report.read_text()); assert report['status']=='complete' and report['final_output']['sha256']==APPROVED_SCAM_SHA
assert_approved_scam_artifacts()
print('사람 승인 후보를 학습 경로로 승격:',scam_path)
print('승인 manifest:',approval_manifest)

In [ ]:
# 4. 데이터 빌드 + 외부 어댑터 + 하드케이스 v5 + 소비된 blind v1~v9 누수 가드
assert_approved_scam_artifacts()
required=[REPO/'data/eval/aihub_train.jsonl',REPO/'data/eval/aihub_real_holdout.jsonl',REPO/'data/eval/beep_real_holdout.jsonl']
assert all(p.exists() for p in required),[str(p) for p in required if not p.exists()]
assert all(hashlib.sha256((REPO/name).read_bytes()).hexdigest()==sha for name,sha in REQUIRED_BUNDLE_FILES.items()), 'eval 정본 SHA 변경'
pan12_training_path=REPO/'data/synthetic/pan12_translated.jsonl'
assert pan12_training_path.is_file(), 'PAN12 정본이 필요합니다. Cell 2의 PAN12_SOURCE_PATH_RAW에 Colab 경로를 입력하고 Cell 2를 다시 실행하세요.'
assert hashlib.sha256(pan12_training_path.read_bytes()).hexdigest()==PAN12_SHA, 'PAN12 정본 SHA 변경'
BLIND_V9=REPO/'tests/fixtures/safe_blind_v9.jsonl'
BLIND_V9_SHA='e98675b8d99588086c348beb8ea4f38c3596894297395e6de4d991691eff48ff'
assert BLIND_V9.exists() and hashlib.sha256(BLIND_V9.read_bytes()).hexdigest()==BLIND_V9_SHA, '회귀 blind v9 정본 누락·변조'
assert not (REPO/'tests/fixtures/safe_blind_v10.jsonl').exists(), 'fresh v10은 후보 고정 전 배치 금지 — Cell 7 완료 후 독립 작성자가 추가해야 함'
consumed_dev_paths=['data/eval/aihub_real_holdout.jsonl','data/eval/beep_real_holdout.jsonl',
    'data/synthetic/emergency/3a/val.jsonl','data/synthetic/emergency/3a/test.jsonl',
    *[f'tests/fixtures/safe_blind_v{i}.jsonl' for i in range(1,10)]]
final_build_cmd=[sys.executable,'scripts/build_final_dataset.py','--synth-repeat','1','--include-aihub-train']
for path in consumed_dev_paths: final_build_cmd+=['--forbidden',path]
hardcase_cmd=[sys.executable,'scripts/build_safe_hardcase_dataset.py','--include-v5','--output','data/synthetic/safe_hardcases_v5/train.jsonl']
for path in consumed_dev_paths: hardcase_cmd+=['--forbidden',path]
steps=[
    [sys.executable,'scripts/download_seed_datasets.py'],
    [sys.executable,'scripts/build_processed_dataset.py'],
    final_build_cmd,
    hardcase_cmd,
    [sys.executable,'scripts/adapt_external_datasets.py'],
]
for cmd in steps: subprocess.run(cmd,cwd=REPO,check=True)
import pandas as pd
from src.data.dedup import find_duplicate_indices
train=pd.read_parquet(REPO/'data/processed/train.parquet')
def norm(x): return re.sub(r'[^0-9a-z가-힣ㄱ-ㅎㅏ-ㅣ]+','',str(x).lower())
train_norm={key for t in train['text'] if (key:=norm(t))}
extra_paths=['data/synthetic/dktc.jsonl','data/synthetic/kmhas.jsonl','data/synthetic/apeach.jsonl',
             'data/synthetic/safe_hardcases_v5/train.jsonl','data/synthetic/scam/train.jsonl']
pan12_path=REPO/'data/synthetic/pan12_translated.jsonl'
assert pan12_path.exists(), 'trainer direct source pan12_translated.jsonl 누락'
extra=[json.loads(x) for x in pan12_path.read_text().splitlines() if x and json.loads(x).get('split_role')=='predator']
for p in extra_paths: extra+=[json.loads(x) for x in (REPO/p).read_text().splitlines() if x]
extra_texts=[x['text'] for x in extra]; extra_norm={key for t in extra_texts if (key:=norm(t))}
development_guard=[]
for name in ['aihub_real_holdout.jsonl','beep_real_holdout.jsonl']:
    development_guard += [json.loads(x) for x in (REPO/'data/eval'/name).read_text().splitlines() if x]
for i in range(1,10):
    development_guard += [json.loads(x) for x in (REPO/'tests/fixtures'/f'safe_blind_v{i}.jsonl').read_text().splitlines() if x]
for split in ['val','test']:
    development_guard += [json.loads(x) for x in (REPO/f'data/synthetic/emergency/3a/{split}.jsonl').read_text().splitlines() if x]
training_texts=[*list(train['text']),*extra_texts]; development_texts=[x['text'] for x in development_guard]
training_norm={key for t in training_texts if (key:=norm(t))}; development_norm={key for t in development_texts if (key:=norm(t))}
assert not (training_norm&development_norm), '실제 학습 입력 ↔ 개발 평가셋 exact leak'
assert not find_duplicate_indices(development_texts,training_texts,threshold=0.8), '실제 학습 입력 ↔ 개발 평가셋 near-dup leak'
print('base train',len(train),'| extra',len(extra),
      '| extra label',pd.Series([x.get('label') for x in extra]).value_counts(dropna=False).to_dict())
assert_approved_scam_artifacts()
print('실 holdout + 소비된 blind v1~v9 + grooming 개발셋 exact/near 누수 가드: OK')

In [ ]:
# 5. 개발 평가 함수 — dev 회귀셋 = 실 holdout + 소비된 blind v1~v9. 규칙보조 OFF.
import numpy as np, yaml
from sklearn.metrics import confusion_matrix
from transformers import AutoTokenizer,AutoModelForSequenceClassification
real=[]
for name in ['aihub_real_holdout.jsonl','beep_real_holdout.jsonl']:
    real += [json.loads(x) for x in (REPO/'data/eval'/name).read_text().splitlines() if x]
dev=[]
for i in range(1,10):
    dev += [json.loads(x) for x in (REPO/'tests/fixtures'/f'safe_blind_v{i}.jsonl').read_text().splitlines() if x]
groom=[]
for split in ['val','test']: groom += [json.loads(x) for x in (REPO/f'data/synthetic/emergency/3a/{split}.jsonl').read_text().splitlines() if x]
def load_predict(path):
    tok=AutoTokenizer.from_pretrained(path); model=AutoModelForSequenceClassification.from_pretrained(path).cuda().eval(); assert model.config.num_labels==2
    def predict(texts,batch=128):
        out=[]
        with torch.inference_mode():
            for i in range(0,len(texts),batch):
                enc=tok(texts[i:i+batch],padding=True,truncation=True,max_length=128,return_tensors='pt').to('cuda')
                out.extend(torch.softmax(model(**enc).logits,dim=-1)[:,1].cpu().tolist())
        return np.array(out)
    return model,predict
sl=lambda key: [i for i,x in enumerate(dev) if str(x['slice'])==key]
fraud=sl('fraud_credentials'); coercive=sl('coercive_control'); ext=sl('digital_extortion')
grooming=[i for i,x in enumerate(dev) if str(x['slice']).startswith('grooming')]
recon=sl('routine_recon')
scam_dev=[i for i,x in enumerate(dev) if str(x['slice']).startswith('scam')]
benign_money=sl('benign_money')
def evaluate_dev(path):
    model,predict=load_predict(path)
    yr=np.array([int(int(x['label'])>0) for x in real]); pr=predict([x['text'] for x in real])
    yd=np.array([x['label'] for x in dev]); pdv=predict([x['text'] for x in dev])
    pg=predict([x['text'] for x in groom])
    candidates=[]
    for adult in np.arange(.40,.86,.01):
        minor=max(.35,round(float(adult)-.16,2))
        rpred=(pr>=adult)
        dthr=np.array([minor if x['receiver_is_minor'] else adult for x in dev]); dpred=(pdv>=dthr)
        tn,fp,fn,tp=confusion_matrix(yr,rpred,labels=[0,1]).ravel(); dtn,dfp,dfn,dtp=confusion_matrix(yd,dpred,labels=[0,1]).ravel()
        row={'adult':round(float(adult),2),'minor':minor,'real_recall':tp/(tp+fn),'real_specificity':tn/(tn+fp),
             'dev_recall':dtp/(dtp+dfn),'dev_specificity':dtn/(dtn+dfp),
             'fraud_recall':float(dpred[fraud].mean()) if fraud else 1.0,'coercive_recall':float(dpred[coercive].mean()) if coercive else 1.0,
             'grooming_recall':float(dpred[grooming].mean()) if grooming else 1.0,'extortion_recall':float(dpred[ext].mean()) if ext else 1.0,
             'recon_recall':float(dpred[recon].mean()) if recon else 1.0,'scam_recall':float(dpred[scam_dev].mean()) if scam_dev else 0.0,
             'benign_money_specificity':float((~dpred[benign_money]).mean()) if benign_money else 0.0,
             'synthetic_grooming_recall':float((pg>=minor).mean())}
        row['pass']=all([row['real_recall']>=.80,row['real_specificity']>=.80,row['dev_recall']>=.85,row['dev_specificity']>=.90,
                         row['fraud_recall']>=.80,row['coercive_recall']>=.80,row['grooming_recall']>=.80,row['extortion_recall']>=.80,
                         row['scam_recall']>=.80,row['benign_money_specificity']>=.875])
        candidates.append(row)
    passing=[x for x in candidates if x['pass']]
    best=max(passing,key=lambda x:(x['real_specificity'],x['dev_specificity'])) if passing else max(candidates,key=lambda x:(min(x['dev_recall'],x['dev_specificity'],x['fraud_recall'],x['coercive_recall']),x['real_specificity']))
    del model; torch.cuda.empty_cache(); return best

In [ ]:
# 6. v7 repeat 1→3 재학습. 개발 게이트 통과 시 즉시 중단
assert torch.cuda.is_available(), '재학습은 GPU 런타임에서만 실행할 수 있습니다.'
print('재학습 GPU:',torch.cuda.get_device_name(0))
assert_approved_scam_artifacts()
ATTEMPTS=[]; SELECTED=None
base=yaml.safe_load((REPO/'configs/module1_binary_hardcases_v7.yaml').read_text())
for repeat in [1,2,3]:
    assert_approved_scam_artifacts()
    cfg=json.loads(json.dumps(base)); name=f'module1_binary_hardcases_v7_r{repeat}'
    cfg['data']['extra_train_repeat']=repeat; cfg['model']['checkpoint_dir']=f'models/checkpoints/{name}'; cfg['paths']['checkpoint_dir']=f'models/checkpoints/{name}'
    temp=Path(tempfile.gettempdir())/f'{name}.yaml'; temp.write_text(yaml.safe_dump(cfg,allow_unicode=True,sort_keys=False))
    subprocess.run([sys.executable,'scripts/train_module1.py','--config',str(temp)],cwd=REPO,check=True)
    ckpt=REPO/cfg['model']['checkpoint_dir']; result=evaluate_dev(ckpt); result.update({'repeat':repeat,'checkpoint':str(ckpt)}); ATTEMPTS.append(result); print(json.dumps(result,ensure_ascii=False,indent=2))
    if result['pass']: SELECTED=result; break
dev_run_date=datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d')
dev_report=next_versioned_path(REPO/'outputs/10_thisabled-ai/보고서'/f'{dev_run_date}_thisabled_사기재학습개발평가.json'); write_json_atomic(dev_report,ATTEMPTS)
assert SELECTED is not None, '3회 모두 개발 게이트 실패 — fresh blind 및 업로드 금지'
assert_approved_scam_artifacts()
def checkpoint_sha(path):
    root=Path(path); digest=hashlib.sha256()
    ignored=('checkpoint-','optimizer','scheduler','trainer_state','rng_state','training_args')
    files=[p for p in root.rglob('*') if p.is_file() and not any(part.startswith(ignored) for part in p.relative_to(root).parts)]
    for p in sorted(files): digest.update(str(p.relative_to(root)).encode()); digest.update(hashlib.sha256(p.read_bytes()).digest())
    return digest.hexdigest()
SELECTED_CHECKPOINT_SHA=checkpoint_sha(SELECTED['checkpoint'])
print('SELECTED',SELECTED,'checkpoint_sha256',SELECTED_CHECKPOINT_SHA,'dev_report',dev_report)
print('후보가 고정됐습니다. 이제 독립 작성자가 safe_blind_v10.jsonl과 정본 SHA를 제공해야 합니다.')

In [ ]:
# 7. 후보 고정 후 독립 작성·봉인된 fresh blind v10 최초 1회 평가 (규칙보조 OFF)
assert_approved_scam_artifacts(); assert checkpoint_sha(SELECTED['checkpoint'])==SELECTED_CHECKPOINT_SHA, '선택 checkpoint 변경 감지'
FRESH_BLIND_EXPECTED_SHA=''  # 독립 작성자가 전달한 safe_blind_v10.jsonl SHA-256 입력
assert re.fullmatch(r'[0-9a-f]{64}',FRESH_BLIND_EXPECTED_SHA), 'fresh blind v10 정본 SHA가 필요합니다.'
REGISTRY_ID='thisabled-ai-fresh-blind-registry-v1'
persistent_audit_dir=Path('/content/drive/MyDrive/thisabled-ai/fresh_blind_registry').resolve()
registry_marker=persistent_audit_dir/'.registry_id'
assert persistent_audit_dir.is_dir() and registry_marker.is_file() and registry_marker.read_text().strip()==REGISTRY_ID, '고정 Drive registry와 .registry_id marker를 먼저 준비해야 합니다.'
assert persistent_audit_dir!=REPO and REPO not in persistent_audit_dir.parents and persistent_audit_dir not in REPO.parents, '감사 폴더는 임시 저장소 및 그 부모 밖의 영속 볼륨이어야 함'
with tempfile.NamedTemporaryFile(dir=persistent_audit_dir,delete=False) as probe: probe.write(b'audit'); probe.flush(); os.fsync(probe.fileno()); probe_path=Path(probe.name)
probe_path.unlink()
blind_path=REPO/'tests/fixtures/safe_blind_v10.jsonl'
assert blind_path.exists(), '독립 작성·봉인된 safe_blind_v10.jsonl 필요'
before=hashlib.sha256(blind_path.read_bytes()).hexdigest(); assert before==FRESH_BLIND_EXPECTED_SHA, 'fresh blind v10 정본 SHA 불일치'
consumption_lock_path=persistent_audit_dir/f'thisabled_freshblindv10_{before}.lock'
blind_run_date=datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d')
blind_output_dir=REPO/'outputs/10_thisabled-ai/보고서'; blind_output_dir.mkdir(parents=True,exist_ok=True)
blind_result_path=blind_output_dir/f'{blind_run_date}_thisabled_freshblindv10평가.json'
blind_ledger_path=blind_output_dir/f'{blind_run_date}_thisabled_freshblindv10소비기록.json'
persistent_blind_path=persistent_audit_dir/f'{blind_run_date}_thisabled_freshblindv10정본.jsonl'
persistent_scam_path=persistent_audit_dir/f'{blind_run_date}_thisabled_승인사기학습데이터.jsonl'
persistent_report_path=persistent_audit_dir/f'{blind_run_date}_thisabled_승인사기검수보고서.json'
persistent_manifest_path=persistent_audit_dir/f'{blind_run_date}_thisabled_사기승인manifest.json'
persistent_checkpoint_path=persistent_audit_dir/f'{blind_run_date}_thisabled_선택모델체크포인트'
persistent_result_path=persistent_audit_dir/f'{blind_run_date}_thisabled_freshblindv10평가.json'
persistent_ledger_path=persistent_audit_dir/f'{blind_run_date}_thisabled_freshblindv10소비기록.json'
prior_local=list(blind_output_dir.glob('*_thisabled_freshblindv10소비기록*.json'))
prior_persistent=list(persistent_audit_dir.glob('*_thisabled_freshblindv10소비기록*.json'))
assert not prior_local and not prior_persistent and not consumption_lock_path.exists(), 'fresh blind v10은 이전 실행에서 이미 소비됨 — 날짜·런타임·폴더를 바꾼 재평가도 금지'
APPROVAL_MANIFEST_SHA=hashlib.sha256(approval_manifest.read_bytes()).hexdigest()
dev_reference_texts=[x['text'] for x in [*real,*dev,*groom]]
DEV_REFERENCE_SHA=hashlib.sha256(json.dumps(dev_reference_texts,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
blind_ledger={'schema_version':2,'status':'started','blind_path':str(blind_path.relative_to(REPO)),'blind_sha256':before,
    'approved_scam_sha256':APPROVED_SCAM_SHA,'verification_report_sha256':APPROVED_REPORT_SHA,
    'approval_manifest_sha256':APPROVAL_MANIFEST_SHA,'source_checkpoint':SELECTED['checkpoint'],
    'checkpoint':str(persistent_checkpoint_path),'checkpoint_sha256':SELECTED_CHECKPOINT_SHA,'repeat':SELECTED['repeat'],
    'thresholds':{'adult':SELECTED['adult'],'minor':SELECTED['minor']},'rule_assist':False,'development_reference_sha256':DEV_REFERENCE_SHA,
    'persistent_blind_path':str(persistent_blind_path),'persistent_scam_path':str(persistent_scam_path),
    'registry_id':REGISTRY_ID,'consumption_lock_path':str(consumption_lock_path),
    'persistent_report_path':str(persistent_report_path),'persistent_manifest_path':str(persistent_manifest_path),
    'result_path':str(persistent_result_path)}
lock_payload={'schema_version':1,'registry_id':REGISTRY_ID,'status':'started','blind_sha256':before,
    'checkpoint_sha256':SELECTED_CHECKPOINT_SHA,'ledger_path':str(persistent_ledger_path)}
try:
    lock_fd=os.open(consumption_lock_path,os.O_WRONLY|os.O_CREAT|os.O_EXCL,0o600)
except FileExistsError as exc:
    raise AssertionError(f'v10 SHA 소비 lock이 이미 존재함: {consumption_lock_path}') from exc
with os.fdopen(lock_fd,'w',encoding='utf-8') as lock_file:
    json.dump(lock_payload,lock_file,ensure_ascii=False,indent=2); lock_file.write('\n'); lock_file.flush(); os.fsync(lock_file.fileno())
write_json_atomic(persistent_ledger_path,blind_ledger); write_json_atomic(blind_ledger_path,blind_ledger)
def copy_file_once(source,target,expected_sha):
    source=Path(source); target=Path(target); assert not target.exists(), f'영속 산출물 충돌: {target}'
    with source.open('rb') as src, tempfile.NamedTemporaryFile(dir=target.parent,delete=False) as temp:
        shutil.copyfileobj(src,temp); temp.flush(); os.fsync(temp.fileno()); staged=Path(temp.name)
    os.replace(staged,target); assert hashlib.sha256(target.read_bytes()).hexdigest()==expected_sha, target
copy_file_once(blind_path,persistent_blind_path,before)
copy_file_once(scam_path,persistent_scam_path,APPROVED_SCAM_SHA)
copy_file_once(verification_report,persistent_report_path,APPROVED_REPORT_SHA)
copy_file_once(approval_manifest,persistent_manifest_path,APPROVAL_MANIFEST_SHA)
assert not persistent_checkpoint_path.exists(), f'영속 checkpoint 충돌: {persistent_checkpoint_path}'
with tempfile.TemporaryDirectory(dir=persistent_audit_dir,prefix='.checkpoint-stage-') as stage:
    staged_checkpoint=Path(stage)/'checkpoint'; shutil.copytree(SELECTED['checkpoint'],staged_checkpoint); os.replace(staged_checkpoint,persistent_checkpoint_path)
assert checkpoint_sha(persistent_checkpoint_path)==SELECTED_CHECKPOINT_SHA, '영속 checkpoint 복사 불일치'
from src.data.scam_synthesis import contains_sensitive_identifier
fresh_rows=[json.loads(x) for x in persistent_blind_path.read_text().splitlines() if x]; fresh_texts=[x['text'] for x in fresh_rows]
required_keys={'id','label','slice','receiver_is_minor','text'}
assert len(fresh_rows)==40 and all(set(x)==required_keys for x in fresh_rows), 'v10은 정확한 40행/5필드 계약이어야 함'
assert sum(type(x['label']) is int and x['label']==0 for x in fresh_rows)==20 and sum(type(x['label']) is int and x['label']==1 for x in fresh_rows)==20, 'v10 label 20/20 필요'
assert len({x['id'] for x in fresh_rows})==40 and len({norm(x['text']) for x in fresh_rows})==40, 'v10 id/text 중복'
assert all(isinstance(x['id'],str) and isinstance(x['slice'],str) and type(x['receiver_is_minor']) is bool and isinstance(x['text'],str) and x['text'].strip() for x in fresh_rows), 'v10 필드 타입 오류'
assert not any(contains_sensitive_identifier(x['text']) for x in fresh_rows), 'v10 실제형 민감 식별자 포함'
slice_counts=collections.Counter(x['slice'] for x in fresh_rows)
required_scam_slices={'scam_impersonation','scam_investment','scam_credential','scam_giftcard','scam_task'}
assert all(slice_counts[s]>=3 for s in required_scam_slices) and slice_counts['benign_money']>=8, f'v10 필수 슬라이스 수량 미달: {slice_counts}'
assert all(x['label']==1 for x in fresh_rows if x['slice'] in required_scam_slices) and all(x['label']==0 for x in fresh_rows if x['slice']=='benign_money'), 'v10 필수 슬라이스 label 오류'
assert not ({norm(t) for t in fresh_texts}&extra_norm), 'fresh blind v10 exact leak vs direct extra sources — v10 폐기 필요'
assert not ({norm(t) for t in fresh_texts}&train_norm), 'fresh blind v10 exact leak vs base train — v10 폐기 필요'
assert not find_duplicate_indices(fresh_texts,extra_texts,threshold=0.8), 'fresh blind v10 near-dup vs direct extra sources — v10 폐기 필요'
assert not find_duplicate_indices(fresh_texts,list(train['text']),threshold=0.8), 'fresh blind v10 near-dup vs base train — v10 폐기 필요'
dev_reference_norm={norm(t) for t in dev_reference_texts}
assert not ({norm(t) for t in fresh_texts}&dev_reference_norm), 'fresh blind v10 exact leak vs development evaluation — v10 폐기 필요'
assert not find_duplicate_indices(dev_reference_texts,fresh_texts,threshold=0.8), 'fresh blind v10 near-dup vs development evaluation — v10 폐기 필요'
result_temp=persistent_audit_dir/f'.{persistent_result_path.name}.tmp'; assert not result_temp.exists() and not persistent_result_path.exists()
subprocess.run([sys.executable,'scripts/evaluate_safe_blind.py','--model',str(persistent_checkpoint_path),'--data',str(persistent_blind_path),
                '--adult-threshold',str(SELECTED['adult']),'--minor-threshold',str(SELECTED['minor']),'--no-rule-assist','--output',str(result_temp)],cwd=REPO,check=True)
assert hashlib.sha256(persistent_blind_path.read_bytes()).hexdigest()==before, 'fresh blind v10 원문 변경 — 평가 무효'
BLIND=json.loads(result_temp.read_text()); expected_thresholds={'adult':SELECTED['adult'],'minor':SELECTED['minor']}
assert BLIND.get('dataset')==str(persistent_blind_path) and BLIND.get('target')==str(persistent_checkpoint_path), '평가 dataset/target 결합 오류'
assert BLIND.get('thresholds')==expected_thresholds and BLIND.get('rule_assist') is False, '평가 threshold/rule 결합 오류'
assert isinstance(BLIND.get('rows'),list) and len(BLIND['rows'])==40 and BLIND.get('overall',{}).get('n')==40, '평가 행 수 오류'
assert all(all(result[k]==source[k] for k in required_keys) and result.get('rule_assist') is False for source,result in zip(fresh_rows,BLIND['rows'],strict=True)), '평가 결과 행 결합 오류'
by=BLIND['by_slice']; assert set(by)==set(slice_counts) and all(by[s]['n']==slice_counts[s] for s in slice_counts), '평가 slice manifest 불일치'
os.replace(result_temp,persistent_result_path)
RESULT_SHA=hashlib.sha256(persistent_result_path.read_bytes()).hexdigest(); copy_file_once(persistent_result_path,blind_result_path,RESULT_SHA)
m=BLIND['overall']; risk_slices={s:by[s]['risk_recall'] for s in by if by[s].get('risk_recall') is not None}
scam_slices={s:risk_slices[s] for s in sorted(required_scam_slices)}
assert by['benign_money'].get('specificity') is not None, 'benign_money specificity 누락'
min_slice=min(risk_slices.values()) if risk_slices else 0.0; benign_money_specificity=by['benign_money']['specificity']
BLIND_PASS=bool(m['risk_recall']>=.80 and m['specificity']>=.90 and min_slice>=.75 and benign_money_specificity>=.875)
blind_ledger.update({'status':'complete','result_sha256':RESULT_SHA,'blind_pass':BLIND_PASS,'overall':m,
    'min_slice_recall':min_slice,'benign_money_specificity':benign_money_specificity})
write_json_atomic(persistent_ledger_path,blind_ledger); write_json_atomic(blind_ledger_path,blind_ledger)
lock_payload.update({'status':'complete','result_sha256':RESULT_SHA,'blind_pass':BLIND_PASS}); write_json_atomic(consumption_lock_path,lock_payload)
print('overall',m); print('scam 슬라이스 recall',scam_slices); print('정상 금전 경계 specificity',benign_money_specificity)
print({'blind_pass':BLIND_PASS,'min_slice_recall':round(min_slice,3),'sha256':before,'영속_ledger':str(persistent_ledger_path)})
assert BLIND_PASS, 'fresh blind v10 실패 — 업로드 금지, v10은 영속 ledger에 소비 완료이며 재사용 금지'

In [ ]:
# 8. 오류 케이스 확인 (FN=놓친 위험 / FP=과플래그)
res=json.loads(blind_result_path.read_text())
for e in res.get('errors',[]):
    kind='FN' if e['label']==1 else 'FP'
    print(kind, e['slice'], round(e.get('risk_prob',0),4), '|', e['text'])

In [ ]:
# 9. 영속 ledger만으로 승인·fresh blind·checkpoint 결합을 재검증한 뒤 명시적으로 HF 업로드
UPLOAD_TO_HF=False
PERSISTENT_LEDGER_PATH_RAW=''  # 재접속 시에도 남아 있는 *_thisabled_freshblindv10소비기록.json
if UPLOAD_TO_HF:
    import collections,hashlib,json,math,os,re,tempfile
    from pathlib import Path
    REGISTRY_ID_U='thisabled-ai-fresh-blind-registry-v1'
    registry_root_u=Path('/content/drive/MyDrive/thisabled-ai/fresh_blind_registry').resolve()
    registry_marker_u=registry_root_u/'.registry_id'
    ledger_path=Path(PERSISTENT_LEDGER_PATH_RAW).expanduser().resolve()
    assert ledger_path.is_file() and ledger_path.parent==registry_root_u, '고정 Drive registry의 영속 ledger만 업로드에 사용할 수 있습니다.'
    assert registry_marker_u.is_file() and registry_marker_u.read_text().strip()==REGISTRY_ID_U, '고정 Drive registry marker 불일치'
    ledger=json.loads(ledger_path.read_text()); assert ledger.get('schema_version')==2 and ledger.get('status')=='complete' and ledger.get('blind_pass') is True
    assert ledger.get('registry_id')==REGISTRY_ID_U and re.fullmatch(r'[0-9a-f]{64}',ledger.get('blind_sha256','')), 'ledger registry/blind SHA 계약 오류'
    lock_path_u=Path(ledger.get('consumption_lock_path','')).expanduser().resolve()
    expected_lock_u=registry_root_u/f"thisabled_freshblindv10_{ledger['blind_sha256']}.lock"
    assert lock_path_u==expected_lock_u and lock_path_u.is_file(), 'v10 SHA 소비 lock 누락 또는 경로 불일치'
    lock_u=json.loads(lock_path_u.read_text())
    assert lock_u.get('schema_version')==1 and lock_u.get('registry_id')==REGISTRY_ID_U and lock_u.get('status')=='complete' and lock_u.get('blind_pass') is True, 'v10 소비 lock 상태 불일치'
    assert lock_u.get('blind_sha256')==ledger['blind_sha256'] and lock_u.get('checkpoint_sha256')==ledger['checkpoint_sha256'] and lock_u.get('result_sha256')==ledger['result_sha256'] and lock_u.get('ledger_path')==str(ledger_path), 'v10 소비 lock ↔ ledger 결합 불일치'
    assert ledger.get('upload_status') is None and not ledger.get('hf_commit_url') and lock_u.get('upload_status') is None and not lock_u.get('hf_commit_url'), 'HF 업로드가 이미 시작·완료됨 — 중복 커밋 금지'
    def write_json_u(path,payload):
        with tempfile.NamedTemporaryFile(mode='w',encoding='utf-8',dir=path.parent,delete=False) as temp:
            json.dump(payload,temp,ensure_ascii=False,indent=2); temp.write('\n'); temp.flush(); os.fsync(temp.fileno()); staged=Path(temp.name)
        os.replace(staged,path)
    def file_sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
    def durable_checkpoint_sha(path):
        root=Path(path); digest=hashlib.sha256(); ignored=('checkpoint-','optimizer','scheduler','trainer_state','rng_state','training_args')
        files=[p for p in root.rglob('*') if p.is_file() and not any(part.startswith(ignored) for part in p.relative_to(root).parts)]
        for p in sorted(files): digest.update(str(p.relative_to(root)).encode()); digest.update(hashlib.sha256(p.read_bytes()).digest())
        return digest.hexdigest()
    blind_path_u=Path(ledger['persistent_blind_path']).resolve(); scam_path_u=Path(ledger['persistent_scam_path']).resolve()
    report_path_u=Path(ledger['persistent_report_path']).resolve(); manifest_path_u=Path(ledger['persistent_manifest_path']).resolve()
    result_path_u=Path(ledger['result_path']).resolve(); checkpoint_path_u=Path(ledger['checkpoint']).resolve()
    assert all(p.parent==registry_root_u for p in [blind_path_u,scam_path_u,report_path_u,manifest_path_u,result_path_u,checkpoint_path_u]), '영속 증거는 고정 Drive registry 직계 경로여야 합니다.'
    assert all(p.exists() for p in [blind_path_u,scam_path_u,report_path_u,manifest_path_u,result_path_u,checkpoint_path_u]), '영속 증거 산출물 누락'
    assert file_sha(blind_path_u)==ledger['blind_sha256'] and file_sha(scam_path_u)==ledger['approved_scam_sha256'], 'blind/scam SHA 불일치'
    assert file_sha(report_path_u)==ledger['verification_report_sha256'] and file_sha(manifest_path_u)==ledger['approval_manifest_sha256'], 'report/manifest SHA 불일치'
    assert file_sha(result_path_u)==ledger['result_sha256'], '평가 결과 SHA 불일치'
    assert durable_checkpoint_sha(checkpoint_path_u)==ledger['checkpoint_sha256'], '업로드 checkpoint 변경'
    manifest_u=json.loads(manifest_path_u.read_text()); report_u=json.loads(report_path_u.read_text()); result_u=json.loads(result_path_u.read_text())
    row_count=sum(1 for line in scam_path_u.read_text().splitlines() if line.strip())
    assert manifest_u.get('human_approved') is True and manifest_u.get('dataset_path')=='data/synthetic/scam/train.jsonl'
    assert manifest_u.get('dataset_sha256')==ledger['approved_scam_sha256'] and manifest_u.get('row_count')==row_count
    assert manifest_u.get('verification_report_sha256')==ledger['verification_report_sha256']
    assert report_u.get('status')=='complete' and report_u.get('final_output',{}).get('sha256')==ledger['approved_scam_sha256'] and report_u['final_output'].get('count')==row_count
    assert result_u.get('dataset')==str(blind_path_u) and result_u.get('target')==str(checkpoint_path_u), 'result dataset/target 결합 불일치'
    assert result_u.get('thresholds')==ledger['thresholds'] and result_u.get('rule_assist') is ledger['rule_assist'] is False, 'result threshold/rule 결합 불일치'
    blind_rows=[json.loads(x) for x in blind_path_u.read_text().splitlines() if x]; result_rows_u=result_u.get('rows')
    input_keys_u={'id','label','slice','receiver_is_minor','text'}; result_keys_u=input_keys_u|{'prediction','risk_prob','rule_assist'}
    assert len(blind_rows)==40 and isinstance(result_rows_u,list) and len(result_rows_u)==40
    assert sum(x['label']==0 for x in blind_rows)==20 and sum(x['label']==1 for x in blind_rows)==20 and len({x['id'] for x in blind_rows})==40
    assert all(set(source)==input_keys_u and set(row)==result_keys_u and all(row[k]==source[k] for k in input_keys_u) for source,row in zip(blind_rows,result_rows_u,strict=True)), 'result.rows ↔ blind strict 결합 실패'
    assert all(type(row['prediction']) is int and row['prediction'] in (0,1) and type(row['risk_prob']) in (int,float) and math.isfinite(row['risk_prob']) and 0<=row['risk_prob']<=1 and row['rule_assist'] is False for row in result_rows_u), '원시 예측 계약 오류'
    assert all(row['prediction']==int(row['risk_prob']>=(ledger['thresholds']['minor'] if row['receiver_is_minor'] else ledger['thresholds']['adult'])) for row in result_rows_u), '원시 예측과 audience threshold 불일치'
    def raw_metrics_u(rows):
        tn=sum(r['label']==0 and r['prediction']==0 for r in rows); fp=sum(r['label']==0 and r['prediction']==1 for r in rows)
        fn=sum(r['label']==1 and r['prediction']==0 for r in rows); tp=sum(r['label']==1 and r['prediction']==1 for r in rows)
        recall=tp/(tp+fn) if tp+fn else None; specificity=tn/(tn+fp) if tn+fp else None
        precision=tp/(tp+fp) if tp+fp else None; npv=tn/(tn+fn) if tn+fn else None
        def f1_u(p,r): return None if p is None or r is None else (2*p*r/(p+r) if p+r else 0.0)
        f1_pos=f1_u(precision,recall); f1_neg=f1_u(npv,specificity)
        return {'n':len(rows),'confusion_matrix':[[tn,fp],[fn,tp]],'risk_recall':recall,'specificity':specificity,
            'fpr':1.0-specificity if specificity is not None else None,'risk_precision':precision,
            'macro_f1':(f1_neg+f1_pos)/2 if f1_neg is not None and f1_pos is not None else None}
    grouped_slice_u=collections.defaultdict(list); grouped_audience_u=collections.defaultdict(list)
    for row in result_rows_u:
        grouped_slice_u[row['slice']].append(row); grouped_audience_u['minor' if row['receiver_is_minor'] else 'adult'].append(row)
    overall_u=raw_metrics_u(result_rows_u); by_u={k:raw_metrics_u(v) for k,v in sorted(grouped_slice_u.items())}
    by_audience_u={k:raw_metrics_u(v) for k,v in sorted(grouped_audience_u.items())}
    errors_u=[row for row in result_rows_u if row['label']!=row['prediction']]
    assert result_u.get('overall')==overall_u==ledger['overall'] and result_u.get('by_slice')==by_u and result_u.get('by_audience')==by_audience_u and result_u.get('errors')==errors_u, '원시 행 재산출 지표 불일치'
    slice_counts_u=collections.Counter(x['slice'] for x in blind_rows); required_scam_u={'scam_impersonation','scam_investment','scam_credential','scam_giftcard','scam_task'}
    assert all(slice_counts_u[s]>=3 for s in required_scam_u) and slice_counts_u['benign_money']>=8
    risk_values_u=[metrics['risk_recall'] for metrics in by_u.values() if metrics.get('risk_recall') is not None]
    assert risk_values_u and by_u['benign_money'].get('specificity') is not None
    min_slice_u=min(risk_values_u); benign_money_u=by_u['benign_money']['specificity']
    recomputed_pass=bool(overall_u.get('risk_recall',0)>=.80 and overall_u.get('specificity',0)>=.90 and min_slice_u>=.75 and benign_money_u>=.875)
    assert recomputed_pass and min_slice_u==ledger['min_slice_recall'] and benign_money_u==ledger['benign_money_specificity'], 'fresh blind 게이트 재검증 실패'
    from getpass import getpass
    from huggingface_hub import login,upload_folder
    token=getpass('HF write token: '); login(token=token,add_to_git_credential=False); del token
    ledger['upload_status']='started'; lock_u['upload_status']='started'; write_json_u(lock_path_u,lock_u); write_json_u(ledger_path,ledger)
    url=upload_folder(repo_id='soyuncj/thisabled-safety-kcelectra',folder_path=checkpoint_path_u,
        commit_message=f"retrain scam-augmented v7 r{ledger['repeat']} blind-v10 approved",
        ignore_patterns=['checkpoint-*','optimizer*','scheduler*','trainer_state*','rng_state*','training_args*'])
    commit_url=str(url); ledger.update({'upload_status':'complete','hf_commit_url':commit_url}); lock_u.update({'upload_status':'complete','hf_commit_url':commit_url}); write_json_u(lock_path_u,lock_u); write_json_u(ledger_path,ledger)
    print('HF_COMMIT_URL:',url)
    print('업로드 후: 새 커밋 SHA를 SAFE_MODEL_REVISION으로 서빙에 반영하고 /health revision 확인.')
else:
    print('업로드 비활성. 재접속 후에도 영속 ledger 경로만 지정하면 독립적으로 재검증할 수 있습니다.')